# Klasifikasi Kualitas Udara - Random Forest

Replikasi model Proyek Akhir:
"SISTEM MONITORING BERBASIS IoT DAN KLASIFIKASI KUALITAS UDARA MENGGUNAKAN RANDOM FOREST"

- **Fitur**: PM2.5, PM10, CO, NO2, O3 (ug/m3)
- **Target**: 5 kategori KLHK
- **Model**: RandomForestClassifier(n_estimators=100, random_state=42)
- **Dataset**: 60% KLHK historis + 15% sensor + 25% sintetis

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib, os
from datetime import datetime, timedelta
from supabase import create_client, Client

SUPABASE_URL = os.getenv("SUPABASE_URL", "")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

if not SUPABASE_URL or not SUPABASE_KEY:
    env_path = ".env"
    if os.path.exists(env_path):
        for line in open(env_path):
            if "=" in line and not line.startswith("#"):
                k, _, v = line.partition("=")
                os.environ[k.strip()] = v.strip().strip(chr(34))
    SUPABASE_URL = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

SUPABASE_URL = SUPABASE_URL.replace("http://", "https://")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f"Connected to {SUPABASE_URL}")

Connected to https://knpwncirbhcytssrxcqx.supabase.co


### 1. Load & Standarisasi Data KLHK Historis

In [2]:
KLHK_DIR = "../klhk/"
FILES = [
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2011.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2012.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2013.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2015.csv",
    "Filedata Data Indeks Standar Pencemaran Udara DKI Jakarta Tahun 2017.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2019.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2020.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2022.csv",
]

all_dfs = []
for f in FILES:
    path = os.path.join(KLHK_DIR, f)
    if not os.path.exists(path):
        print(f"SKIP: {f}")
        continue
    df = pd.read_csv(path)
    df["_src"] = f[:4]
    all_dfs.append(df)
    print(f"Loaded {f}: {len(df)} rows, cols={list(df.columns[:8])}")

raw = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal KLHK: {len(raw)} baris")

Loaded Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2011.csv: 365 rows, cols=['periode_data', 'tanggal', 'pm10', 'so2', 'co', 'o3', 'no2', 'max']
Loaded Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2012.csv: 366 rows, cols=['periode_data', 'tanggal', 'pm10', 'so2', 'co', 'o3', 'no2', 'max']
Loaded Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2013.csv: 365 rows, cols=['periode_data', 'tanggal', 'pm10', 'so2', 'co', 'o3', 'no2', 'max']
Loaded Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2015.csv: 365 rows, cols=['periode_data', 'tanggal', 'pm10', 'so2', 'co', 'o3', 'no2', 'max']
Loaded Filedata Data Indeks Standar Pencemaran Udara DKI Jakarta Tahun 2017.csv: 1825 rows, cols=['periode_data', 'tanggal', 'wilayah', 'pm10', 'so2', 'co', 'o3', 'no2']
Loaded Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2019.csv: 345 rows, cols=['periode_data', 'tanggal', 'pm10', 'so2', 'co', 'o3', 'no2', 'max']
Loaded Filedata Indeks Standar Pencemaran Udara (ISPU) 

### 2. Standarisasi Kolom + Reverse ISPI ke ug/m3

In [3]:
LABEL_MAP = {
    "BAIK": "Baik", "SEDANG": "Sedang", "TIDAK SEHAT": "Tidak Sehat",
    "SANGAT TIDAK SEHAT": "Sangat Tidak Sehat", "BERBAHAYA": "Berbahaya",
}

def ispi_to_conc(ispi, bp):
    if ispi <= 0: return 0
    for cl, ch, il, ih in bp:
        if ispi <= ih:
            return cl + (ispi - il) / (ih - il) * (ch - cl)
    return bp[-1][1]

BP_PM25 = [(0,15,0,50),(15,35,50,100),(35,55,100,200),(55,150,200,300),(150,250,300,400),(250,350,400,500)]
BP_PM10 = [(0,50,0,50),(50,150,50,100),(150,350,100,200),(350,420,200,300),(420,500,300,400),(500,600,400,500)]
BP_CO   = [(0,5000,0,50),(5000,10000,50,100),(10000,17000,100,200),(17000,34000,200,300),(34000,46000,300,400),(46000,56000,400,500)]
BP_NO2  = [(0,40,0,50),(40,80,50,100),(80,180,100,200),(180,280,200,300),(280,565,300,400),(565,665,400,500)]
BP_O3   = [(0,60,0,50),(60,120,50,100),(120,180,100,200),(180,240,200,300),(240,400,300,500)]

O3_CONV = 1.963  # ppb to ug/m3

rows = []
for _, r in raw.iterrows():
    cat_col = "kategori" if "kategori" in raw.columns and "categori" not in raw.columns else "categori"
    if cat_col not in r or pd.isna(r.get(cat_col)): continue
    label = LABEL_MAP.get(str(r[cat_col]).strip().upper())
    if not label: continue

    pm10_ispi = pd.to_numeric(r.get("pm10", r.get("pm_10", 0)), errors="coerce")
    co_ispi   = pd.to_numeric(r.get("co", 0), errors="coerce")
    o3_ispi   = pd.to_numeric(r.get("o3", 0), errors="coerce")
    no2_ispi  = pd.to_numeric(r.get("no2", 0), errors="coerce")

    if pd.notna(r.get("pm_duakomalima")):
        pm25_ispi = pd.to_numeric(r["pm_duakomalima"], errors="coerce")
    else:
        pm25_ispi = pm10_ispi * 0.7

    pm25 = ispi_to_conc(pm25_ispi if pd.notna(pm25_ispi) else 0, BP_PM25)
    pm10 = ispi_to_conc(pm10_ispi if pd.notna(pm10_ispi) else 0, BP_PM10)
    co   = ispi_to_conc(co_ispi if pd.notna(co_ispi) else 0, BP_CO)
    no2  = ispi_to_conc(no2_ispi if pd.notna(no2_ispi) else 0, BP_NO2)
    o3_ppb = ispi_to_conc(o3_ispi if pd.notna(o3_ispi) else 0, BP_O3)
    o3   = o3_ppb * O3_CONV

    if pm25 > 0 and pm10 > 0:
        rows.append({"PM2.5": pm25, "PM10": pm10, "CO": co, "NO2": no2, "O3": o3, "Label": label, "src": "klhk"})

df_klhk = pd.DataFrame(rows)
print(f"KLHK clean: {len(df_klhk)} baris")
print(df_klhk["Label"].value_counts())
print(f"\nSample:")
print(df_klhk.head(3).to_string())

KLHK clean: 2141 baris
Label
Sedang                1105
Tidak Sehat            790
Sangat Tidak Sehat     156
Baik                    89
Berbahaya                1
Name: count, dtype: int64

Sample:
   PM2.5  PM10      CO   NO2        O3   Label   src
0  14.91  92.0  5400.0  24.8  207.2928  Sedang  klhk
1  12.81  72.0  3500.0  23.2  230.8488  Sedang  klhk
2  14.49  88.0  3100.0  22.4  162.5364  Sedang  klhk


### 3. Load Data Sensor dari Supabase

In [4]:
TABLE = "tb_konsentrasi_gas"
since = (datetime.utcnow() - timedelta(days=90)).isoformat()
all_data, offset = [], 0
while True:
    resp = supabase.table(TABLE) \
        .select("pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,created_at") \
        .gte("created_at", since) \
        .order("created_at", desc=False).range(offset, offset+999).execute()
    if not resp.data: break
    all_data.extend(resp.data)
    offset += len(resp.data)

df_sensor = pd.DataFrame(all_data)
for c in ["pm25_ugm3","pm10_ugm3","co_ugm3","no2_ugm3","o3_ugm3"]:
    df_sensor[c] = pd.to_numeric(df_sensor[c], errors="coerce")
df_sensor = df_sensor.dropna(subset=["pm25_ugm3","pm10_ugm3","o3_ugm3"])

def conc_to_ispi(val, bp):
    if val <= 0: return 0
    for cl, ch, il, ih in bp:
        if val <= ch: return il + (val - cl) / (ch - cl) * (ih - il)
    return bp[-1][3]

def ispi_to_label(ispi):
    if ispi <= 50: return "Baik"
    if ispi <= 100: return "Sedang"
    if ispi <= 200: return "Tidak Sehat"
    if ispi <= 300: return "Sangat Tidak Sehat"
    return "Berbahaya"

sensor_rows = []
for _, r in df_sensor.iterrows():
    pm25, pm10, co, no2 = r["pm25_ugm3"], r["pm10_ugm3"], r["co_ugm3"], r["no2_ugm3"]
    o3_ugm3 = r["o3_ugm3"]
    o3_ppb = o3_ugm3 / O3_CONV
    ispis = [conc_to_ispi(pm25,BP_PM25), conc_to_ispi(pm10,BP_PM10), conc_to_ispi(co,BP_CO), conc_to_ispi(no2,BP_NO2), conc_to_ispi(o3_ppb,BP_O3)]
    label = ispi_to_label(max(ispis))
    sensor_rows.append({"PM2.5": pm25, "PM10": pm10, "CO": co, "NO2": no2, "O3": o3_ugm3, "Label": label, "src": "sensor"})

df_sensor_clean = pd.DataFrame(sensor_rows)
print(f"Sensor clean: {len(df_sensor_clean)} baris")
print(df_sensor_clean["Label"].value_counts())

Sensor clean: 13535 baris
Label
Berbahaya             7840
Sedang                5112
Baik                   337
Tidak Sehat            245
Sangat Tidak Sehat       1
Name: count, dtype: int64


### 4. Data Sintetis (Baik + Berbahaya + Sangat Tidak Sehat)

In [5]:
np.random.seed(42)

SYNTH = {
    "Baik": {"PM2.5": (1,14), "PM10": (5,48), "CO": (200,4800), "NO2": (1,38), "O3": (2,55*O3_CONV)},
    "Sedang": {"PM2.5": (16,34), "PM10": (51,148), "CO": (5100,9800), "NO2": (41,78), "O3": (61*O3_CONV,119*O3_CONV)},
    "Tidak Sehat": {"PM2.5": (36,54), "PM10": (151,348), "CO": (10100,16800), "NO2": (81,178), "O3": (121*O3_CONV,179*O3_CONV)},
    "Sangat Tidak Sehat": {"PM2.5": (56,149), "PM10": (351,419), "CO": (17100,33800), "NO2": (181,279), "O3": (181*O3_CONV,239*O3_CONV)},
    "Berbahaya": {"PM2.5": (151,350), "PM10": (421,550), "CO": (34100,50000), "NO2": (281,500), "O3": (241*O3_CONV,380*O3_CONV)},
}

synth_rows = []
for label, ranges in SYNTH.items():
    n = 800 if label in ("Baik", "Berbahaya", "Sangat Tidak Sehat") else 300
    for _ in range(n):
        synth_rows.append({
            "PM2.5": np.random.uniform(*ranges["PM2.5"]),
            "PM10": np.random.uniform(*ranges["PM10"]),
            "CO": np.random.uniform(*ranges["CO"]),
            "NO2": np.random.uniform(*ranges["NO2"]),
            "O3": np.random.uniform(*ranges["O3"]),
            "Label": label, "src": "synthetic"
        })

df_synth = pd.DataFrame(synth_rows)
print(f"Sintetis: {len(df_synth)} baris")
print(df_synth["Label"].value_counts())

Sintetis: 1050 baris
Label
Baik                  300
Berbahaya             300
Sedang                150
Tidak Sehat           150
Sangat Tidak Sehat    150
Name: count, dtype: int64


### 5. Gabungkan, Train & Evaluasi

In [6]:
df_all = pd.concat([df_klhk, df_sensor_clean, df_synth], ignore_index=True)
FEATURES = ["PM2.5", "PM10", "CO", "NO2", "O3"]
LABELS = ["Baik", "Sedang", "Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]

X = df_all[FEATURES]
y = df_all["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Dataset: {len(df_all)} baris")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"\nDistribusi label:")
print(y.value_counts().to_string())

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=LABELS))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS).to_string())

importance = pd.DataFrame({"Feature": FEATURES, "Importance": model.feature_importances_}).sort_values("Importance", ascending=False)
print(f"\nFeature Importance:")
print(importance.to_string(index=False))

Dataset: 16726 baris
Train: 12544, Test: 4182

Distribusi label:
Label
Berbahaya             8141
Sedang                6367
Tidak Sehat           1185
Baik                   726
Sangat Tidak Sehat     307

Accuracy: 0.9971 (99.71%)

Classification Report:
                    precision    recall  f1-score   support

              Baik       0.98      0.99      0.99       182
            Sedang       1.00      1.00      1.00      2035
       Tidak Sehat       1.00      0.99      0.99        77
Sangat Tidak Sehat       1.00      1.00      1.00      1592
         Berbahaya       0.99      0.98      0.99       296

          accuracy                           1.00      4182
         macro avg       0.99      0.99      0.99      4182
      weighted avg       1.00      1.00      1.00      4182

Confusion Matrix:
                    Baik  Sedang  Tidak Sehat  Sangat Tidak Sehat  Berbahaya
Baik                 180       2            0                   0          0
Sedang                 3    

### 6. Simpan Model & Quick Test

In [7]:
joblib.dump(model, "random_forest_air_quality.pkl")
print("Model saved: random_forest_air_quality.pkl")

test_sample = pd.DataFrame([{
    "PM2.5": 22.5,
    "PM10": 85.3,
    "CO": 3200,
    "NO2": 18.2,
    "O3": 45.6
}])

pred = model.predict(test_sample)[0]
prob = model.predict_proba(test_sample)[0]
print(f"\nQuick Test:")
print(f"  Prediksi: {pred}")
for i, cls in enumerate(model.classes_):
    print(f"  {cls:25s}: {prob[i]*100:.1f}%")

Model saved: random_forest_air_quality.pkl

Quick Test:
  Prediksi: Sedang
  Baik                     : 2.0%
  Berbahaya                : 0.0%
  Sangat Tidak Sehat       : 0.0%
  Sedang                   : 61.0%
  Tidak Sehat              : 37.0%
